## Capture-24 Data Preprocessing notebook
### Step 0. Setup the paths and env variables

In [8]:
# =========================
# STEP 0 — Setup & contracts
# =========================
from pathlib import Path
import json, sys, re
import numpy as np
import pandas as pd
from tqdm import tqdm

ROOT = Path("/home/aidan/IMU_LM_Data")
sys.path.insert(0, str(ROOT))

from UTILS.helpers import (
    resample_df,            # decimate FIR-based: (df, target_cols, factor)
    normalize_str, keyize, _keyize
)

BASE    = ROOT / "data"
RAW     = BASE / "raw_data"
CLEANED = BASE / "cleaned_premerge"
MERGED  = BASE / "merged_dataset"

# Capture-24 raw root (edit if your internal folder name differs)
RAW_CAPTURE = RAW / "Capture-24" / "capture24"

# Schema + global mapping contracts
SCHEMA_PATH       = ROOT / "Unification" / "schemas" / "continuous_stream_schema.json"
ACTIVITY_MAP_PATH = ROOT / "Unification" / "schemas" / "activity_mapping.json"

SCHEMA       = json.loads(SCHEMA_PATH.read_text())
ACT_MAP_FULL = json.loads(ACTIVITY_MAP_PATH.read_text())

UNKNOWN_ID = int(ACT_MAP_FULL.get("unknown_activity_id", 9000))
ID2NAME    = {int(x["id"]): x["name"] for x in ACT_MAP_FULL["label_set"]}

print("Paths & contracts ready.")
print("RAW_CAPTURE:", RAW_CAPTURE)
print("CLEANED    :", CLEANED)
print("Schema rate :", SCHEMA["rate_hz"])
print("Unknown ID  :", UNKNOWN_ID)


Paths & contracts ready.
RAW_CAPTURE: /home/aidan/IMU_LM_Data/data/raw_data/Capture-24/capture24
CLEANED    : /home/aidan/IMU_LM_Data/data/cleaned_premerge
Schema rate : 50
Unknown ID  : 9000


### Step 1. Ingest, preporccess and map the data 

In [9]:
# ==========================================
# STEP 1 — Ingest Capture-24, normalize to 50Hz  (FIXED)
#   - Adds dataset column early
#   - Adds gyro placeholders early (NaN)
#   - Robust unlabeled detection for pandas string dtype
# ==========================================
import re
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path

UNLABELED_ID = 9000
UNLABELED_LABEL = "unknown"

def extract_top_level_label(s: str):
    if pd.isna(s):
        return pd.NA
    return str(s).split(";", 1)[0].strip()

def parse_capture24_cpa_code(s: str):
    m = re.search(r"\b(\d{4,6})\b", str(s))
    return int(m.group(1)) if m else None

def load_capture24_raw(
    raw_capture_dir: Path,
    downsample_to_50hz: bool = True,
    debug_one_subject: bool = False,
    debug_subject_id: str = "P001",
    preview_n: int = 3,
) -> pd.DataFrame:
    files = sorted(raw_capture_dir.glob("P*.csv.gz"))
    if not files:
        raise FileNotFoundError(f"No P*.csv.gz found under: {raw_capture_dir}")

    if debug_one_subject:
        one = raw_capture_dir / f"{debug_subject_id}.csv.gz"
        if not one.exists():
            raise FileNotFoundError(f"Debug subject file not found: {one}")
        files = [one]
        print(f"[DEBUG] Processing ONLY: {one.name}")

    all_rows = []
    g_const = 9.80665

    for f in tqdm(files, desc="CAPTURE-24 subjects"):
        df = pd.read_csv(f, compression="gzip", low_memory=False, dtype={"annotation": "string"})
        df = df.rename(columns={"x": "acc_x", "y": "acc_y", "z": "acc_z"})

        # ---- identity ----
        df["dataset"] = "capture24"
        df["subject_id"] = f.stem.replace(".csv", "")
        df["session_id"] = "day1"

        # ---- gyro placeholders (Capture-24 has none) ----
        df["gyro_x"] = np.nan
        df["gyro_y"] = np.nan
        df["gyro_z"] = np.nan

        # ---- time -> timestamp_ns ----
        df["time_dt"] = pd.to_datetime(df["time"], errors="coerce")
        df = df.dropna(subset=["time_dt"]).sort_values("time_dt").reset_index(drop=True)

        t0 = df["time_dt"].iloc[0]
        df["timestamp_ns"] = (df["time_dt"] - t0).astype("timedelta64[ns]").astype("int64")

        # ---- units sanity: if looks like "g", convert to m/s^2 ----
        mag_med = np.nanmedian(np.sqrt(df["acc_x"]**2 + df["acc_y"]**2 + df["acc_z"]**2))
        if 0.5 <= mag_med <= 2.5:
            df[["acc_x", "acc_y", "acc_z"]] = df[["acc_x", "acc_y", "acc_z"]].astype(np.float64) * g_const

        # ---- labels ----
        df["annotation"] = df["annotation"].astype("string")

        # robust unlabeled detection: NA or empty/whitespace
        ann_str = df["annotation"].astype("string")
        unlabeled = ann_str.isna() | (ann_str.str.strip() == "")

        df["dataset_activity_label"] = df["annotation"].map(extract_top_level_label).astype("string")
        df["dataset_activity_id_tmp"] = df["annotation"].map(parse_capture24_cpa_code)

        df.loc[unlabeled, "dataset_activity_label"] = UNLABELED_LABEL
        df.loc[unlabeled, "dataset_activity_id_tmp"] = UNLABELED_ID

        # ---- downsample 100Hz -> 50Hz ----
        if downsample_to_50hz:
            labels = df[[
                "annotation",
                "dataset_activity_label",
                "dataset_activity_id_tmp"
            ]].copy()

            target_cols = ["acc_x", "acc_y", "acc_z"]
            df_sig = resample_df(df, target_cols=target_cols, factor=2).reset_index(drop=True)

            # align labels by exact decimation
            labels_ds = labels.iloc[::2].reset_index(drop=True)
            labels_ds = labels_ds.iloc[:len(df_sig)].reset_index(drop=True)

            df = df_sig
            df[["annotation", "dataset_activity_label", "dataset_activity_id_tmp"]] = labels_ds[
                ["annotation", "dataset_activity_label", "dataset_activity_id_tmp"]
            ]

            # enforce strict 20ms grid
            df["timestamp_ns"] = np.arange(len(df), dtype=np.int64) * 20_000_000

            # keep identity + gyro placeholders after resample_df (depends on helper behavior)
            df["dataset"] = "capture24"
            df["subject_id"] = f.stem.replace(".csv", "")
            df["session_id"] = "day1"
            df["gyro_x"] = np.nan
            df["gyro_y"] = np.nan
            df["gyro_z"] = np.nan

        all_rows.append(df[[
            "dataset",
            "subject_id", "session_id", "timestamp_ns",
            "acc_x", "acc_y", "acc_z",
            "gyro_x", "gyro_y", "gyro_z",
            "annotation", "dataset_activity_label", "dataset_activity_id_tmp"
        ]])

    raw = pd.concat(all_rows, ignore_index=True)

    # ---- deterministic fallback native IDs when CPA code missing but annotation exists ----
    missing_cpa = raw["dataset_activity_id_tmp"].isna()
    has_ann = raw["annotation"].notna() & (raw["annotation"].astype("string").str.strip() != "")
    need_neg = missing_cpa & has_ann
    if need_neg.any():
        uniq = sorted(raw.loc[need_neg, "dataset_activity_label"].astype(str).unique())
        neg_ids = {lab: -(i + 1) for i, lab in enumerate(uniq)}
        raw.loc[need_neg, "dataset_activity_id_tmp"] = raw.loc[need_neg, "dataset_activity_label"].map(
            lambda s: neg_ids[str(s)]
        )

    raw["dataset_activity_id"] = raw["dataset_activity_id_tmp"].astype("int16")
    raw = raw.drop(columns=["dataset_activity_id_tmp"])

    print("\n=== RAW SUMMARY (CAPTURE-24 wrist) ===")
    print(f"Files: {len(files)} | Rows: {raw.shape[0]:,} | Cols: {raw.shape[1]}")
    print("\nTop dataset_activity_label (top-level):")
    print(raw["dataset_activity_label"].value_counts(dropna=False).head(15))
    cpa_rate = (raw["dataset_activity_id"] > 0).mean() * 100
    print(f"\nCPA-coded rows: {cpa_rate:.2f}% (positive IDs)")

    if preview_n and preview_n > 0:
        print("\nPreview:")
        print(raw.head(preview_n).to_string(index=False))

    return raw

raw_capture24 = load_capture24_raw(RAW_CAPTURE, debug_one_subject=True, debug_subject_id="P001")
raw_capture24.head(5)


[DEBUG] Processing ONLY: P001.csv.gz


CAPTURE-24 subjects: 100%|██████████| 1/1 [00:21<00:00, 21.62s/it]



=== RAW SUMMARY (CAPTURE-24 wrist) ===
Files: 1 | Rows: 5,010,001 | Cols: 13

Top dataset_activity_label (top-level):
dataset_activity_label
7030 sleeping     1371002
unknown           1275121
home activity     1204418
leisure            714985
transportation     435374
mixed-activity       9101
Name: count, dtype: Int64

CPA-coded rows: 99.82% (positive IDs)

Preview:
  dataset subject_id session_id  timestamp_ns     acc_x     acc_y    acc_z  gyro_x  gyro_y  gyro_z             annotation dataset_activity_label  dataset_activity_id
capture24       P001       day1             0 -3.433019 -3.920670 4.844961     NaN     NaN     NaN 7030 sleeping;MET 0.95          7030 sleeping                 7030
capture24       P001       day1      20000000 -4.885357 -5.587248 6.891190     NaN     NaN     NaN 7030 sleeping;MET 0.95          7030 sleeping                 7030
capture24       P001       day1      40000000 -4.420409 -5.043319 6.239621     NaN     NaN     NaN 7030 sleeping;MET 0.95        

,dataset,subject_id,session_id,timestamp_ns,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,annotation,dataset_activity_label,dataset_activity_id
0,capture24,P001,day1,0,-3.433019,-3.920670,4.844961,NaN,NaN,NaN,7030 sleeping;MET 0.95,7030 sleeping,7030
1,capture24,P001,day1,20000000,-4.885357,-5.587248,6.891190,NaN,NaN,NaN,7030 sleeping;MET 0.95,7030 sleeping,7030
2,capture24,P001,day1,40000000,-4.420409,-5.043319,6.239621,NaN,NaN,NaN,7030 sleeping;MET 0.95,7030 sleeping,7030
3,capture24,P001,day1,60000000,-4.677948,-5.380952,6.595746,NaN,NaN,NaN,7030 sleeping;MET 0.95,7030 sleeping,7030
4,capture24,P001,day1,80000000,-4.511671,-5.249894,6.375601,NaN,NaN,NaN,7030 sleeping;MET 0.95,7030 sleeping,7030


### Step 2. Map the data and audit the mapping

In [10]:
# # ============================================
# # STEP 2 — Map + audit: dataset_activity_label -> global_id (via activity_mapping.json)
# #   - schema is the ONLY source of truth
# #   - Step 1 already guarantees no NA labels (unknown/9000)
# # ============================================
# if raw_capture24.empty:
#     raise SystemExit("No CAPTURE-24 rows after loading. Check RAW_CAPTURE path/layout.")

# MAPPING   = ACT_MAP_FULL["mapping"]  # normalized str -> global id
# UNKNOWN_ID = int(ACT_MAP_FULL.get("unknown_activity_id", 9000))

# def dataset_to_global_id(ds_label: str) -> int:
#     k = normalize_str(str(ds_label))
#     return int(MAPPING.get(k, UNKNOWN_ID))

# raw_capture24["global_activity_id"] = raw_capture24["dataset_activity_label"].map(dataset_to_global_id).astype("int16")
# raw_capture24["global_activity_label"] = raw_capture24["global_activity_id"].map(
#     lambda x: ID2NAME.get(int(x), "other")
# ).astype("string")

# # Audit
# cov = (raw_capture24["global_activity_id"] != UNKNOWN_ID).mean() * 100
# print(f"Global mapping coverage: {cov:.2f}% (unknown={UNKNOWN_ID})")

# print("\nTop dataset_activity_label:")
# print(raw_capture24["dataset_activity_label"].value_counts().head(15))

# print("\nTop global_activity_label:")
# print(raw_capture24["global_activity_label"].value_counts().head(15))

# unmapped = raw_capture24.loc[raw_capture24["global_activity_id"] == UNKNOWN_ID, "dataset_activity_label"] \
#     .value_counts().head(25)
# print("\nUnmapped dataset_activity_label (top):")
# print(unmapped)

# raw_capture24.head(10)
# ============================================
# STEP 2 — Map + audit (FIXED): dataset_activity_label -> global_id
#   - Pre-normalize mapping keys once
#   - Audit by unique label counts
# ============================================
if raw_capture24.empty:
    raise SystemExit("No CAPTURE-24 rows after loading. Check RAW_CAPTURE path/layout.")

UNKNOWN_ID = int(ACT_MAP_FULL.get("unknown_activity_id", 9000))

# Pre-normalize mapping keys once (critical fix)
MAPPING_RAW  = ACT_MAP_FULL.get("mapping", {})
MAPPING_NORM = { normalize_str(k): int(v) for k, v in MAPPING_RAW.items() }

def dataset_to_global_id(ds_label: str) -> int:
    k = normalize_str(ds_label)
    return int(MAPPING_NORM.get(k, UNKNOWN_ID))

# Apply mapping
raw_capture24["global_activity_id"] = (
    raw_capture24["dataset_activity_label"]
      .astype("string")
      .map(dataset_to_global_id)
      .astype("int16")
)

raw_capture24["global_activity_label"] = (
    raw_capture24["global_activity_id"]
      .map(lambda x: ID2NAME.get(int(x), "other"))
      .astype("string")
)

# ---- AUDIT (by unique label counts) ----
raw_counts = (
    raw_capture24["dataset_activity_label"]
      .astype("string")
      .map(normalize_str)
      .value_counts(dropna=False)
      .rename_axis("raw_label_norm")
      .reset_index(name="count")
)

raw_counts["mapped_id"] = raw_counts["raw_label_norm"].map(MAPPING_NORM).fillna(UNKNOWN_ID).astype(int)
raw_counts["mapped_nm"] = raw_counts["mapped_id"].map(lambda x: ID2NAME.get(int(x), "other"))

unmapped = raw_counts.loc[raw_counts["mapped_id"] == UNKNOWN_ID]
mapped_ct = int(raw_counts.loc[raw_counts["mapped_id"] != UNKNOWN_ID, "count"].sum())
total_ct  = int(raw_counts["count"].sum())
cov_pct   = 100.0 * mapped_ct / max(total_ct, 1)

print(f"[CAP24 STEP2] coverage={cov_pct:.2f}%  (mapped={mapped_ct:,} / total={total_ct:,})  unknown={UNKNOWN_ID}")
print(f"[CAP24 STEP2] unique_labels={len(raw_counts)}  unmapped_unique={len(unmapped)}")

print("\nTop dataset_activity_label (raw):")
print(raw_capture24["dataset_activity_label"].value_counts(dropna=False).head(15))

print("\nTop global_activity_label:")
print(raw_capture24["global_activity_label"].value_counts(dropna=False).head(15))

print("\nUnmapped dataset_activity_label (top 25):")
print(unmapped.sort_values("count", ascending=False).head(25).to_string(index=False))

raw_capture24.head(10)


[CAP24 STEP2] coverage=74.55%  (mapped=3,734,880 / total=5,010,001)  unknown=9000
[CAP24 STEP2] unique_labels=6  unmapped_unique=1

Top dataset_activity_label (raw):
dataset_activity_label
7030 sleeping     1371002
unknown           1275121
home activity     1204418
leisure            714985
transportation     435374
mixed-activity       9101
Name: count, dtype: Int64

Top global_activity_label:
global_activity_label
posture_stationary       2521361
other                    1275121
adl_household_general    1204418
adl_personal_care           9101
Name: count, dtype: Int64

Unmapped dataset_activity_label (top 25):
raw_label_norm   count  mapped_id mapped_nm
       unknown 1275121       9000     other


,dataset,subject_id,session_id,timestamp_ns,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,annotation,dataset_activity_label,dataset_activity_id,global_activity_id,global_activity_label
0,capture24,P001,day1,0,-3.433019,-3.920670,4.844961,NaN,NaN,NaN,7030 sleeping;MET 0.95,7030 sleeping,7030,1,posture_stationary
1,capture24,P001,day1,20000000,-4.885357,-5.587248,6.891190,NaN,NaN,NaN,7030 sleeping;MET 0.95,7030 sleeping,7030,1,posture_stationary
2,capture24,P001,day1,40000000,-4.420409,-5.043319,6.239621,NaN,NaN,NaN,7030 sleeping;MET 0.95,7030 sleeping,7030,1,posture_stationary
3,capture24,P001,day1,60000000,-4.677948,-5.380952,6.595746,NaN,NaN,NaN,7030 sleeping;MET 0.95,7030 sleeping,7030,1,posture_stationary
4,capture24,P001,day1,80000000,-4.511671,-5.249894,6.375601,NaN,NaN,NaN,7030 sleeping;MET 0.95,7030 sleeping,7030,1,posture_stationary
5,capture24,P001,day1,100000000,-4.727792,-5.427804,6.515527,NaN,NaN,NaN,7030 sleeping;MET 0.95,7030 sleeping,7030,1,posture_stationary
6,capture24,P001,day1,120000000,-4.729825,-5.259544,6.427966,NaN,NaN,NaN,7030 sleeping;MET 0.95,7030 sleeping,7030,1,posture_stationary
7,capture24,P001,day1,140000000,-4.702888,-5.216122,6.480389,NaN,NaN,NaN,7030 sleeping;MET 0.95,7030 sleeping,7030,1,posture_stationary
8,capture24,P001,day1,160000000,-4.565061,-5.288339,6.526506,NaN,NaN,NaN,7030 sleeping;MET 0.95,7030 sleeping,7030,1,posture_stationary
9,capture24,P001,day1,180000000,-4.586475,-5.266580,6.466683,NaN,NaN,NaN,7030 sleeping;MET 0.95,7030 sleeping,7030,1,posture_stationary


### Step 3. Build and clean dataset in stream json fromat

In [11]:
# =========================================================
# STEP 3 — Build schema-ordered continuous_stream (v3) df
# =========================================================
def to_continuous_stream_capture24(df_raw: pd.DataFrame, dataset_name: str = "capture24") -> pd.DataFrame:
    if df_raw.empty:
        return pd.DataFrame(columns=[c["name"] for c in SCHEMA["columns"]])

    out = pd.DataFrame({
        "dataset":              dataset_name,
        "subject_id":           df_raw["subject_id"].astype("string"),
        "session_id":           df_raw["session_id"].astype("string"),
        "timestamp_ns":         df_raw["timestamp_ns"].astype("int64"),

        "acc_x": df_raw["acc_x"].astype("float32"),
        "acc_y": df_raw["acc_y"].astype("float32"),
        "acc_z": df_raw["acc_z"].astype("float32"),

        # No gyro in CAPTURE-24
        "gyro_x": df_raw["gyro_x"].astype("float32"),
        "gyro_y": df_raw["gyro_y"].astype("float32"),
        "gyro_z": df_raw["gyro_z"].astype("float32"),

        "global_activity_id":    df_raw["global_activity_id"].astype("int16"),
        "global_activity_label": df_raw["global_activity_label"].astype("string"),

        "dataset_activity_id":   df_raw["dataset_activity_id"].astype("int16"),
        "dataset_activity_label":df_raw["dataset_activity_label"].astype("string"),
    })

    order = [c["name"] for c in SCHEMA["columns"]]
    return out[order]

capture24_df = to_continuous_stream_capture24(raw_capture24, dataset_name="capture24")
print("UNIFIED rows:", len(capture24_df))
capture24_df.head(3)


UNIFIED rows: 5010001


,dataset,subject_id,session_id,timestamp_ns,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,global_activity_id,global_activity_label,dataset_activity_id,dataset_activity_label
0,capture24,P001,day1,0,-3.433019,-3.920670,4.844961,NaN,NaN,NaN,1,posture_stationary,7030,7030 sleeping
1,capture24,P001,day1,20000000,-4.885357,-5.587247,6.891191,NaN,NaN,NaN,1,posture_stationary,7030,7030 sleeping
2,capture24,P001,day1,40000000,-4.420410,-5.043319,6.239621,NaN,NaN,NaN,1,posture_stationary,7030,7030 sleeping


### Step 4. Audit check the unified frame

In [12]:
# ==========================================
# STEP 4 — Contract checks & quick QA
# ==========================================
print("Subjects:", capture24_df["subject_id"].nunique(),
      "| Sessions:", capture24_df["session_id"].nunique())

# Monotonic timestamp per (subject, session)
viol = 0
for (_sid, _sess), g in capture24_df.groupby(["subject_id","session_id"], sort=False):
    ts = g["timestamp_ns"].to_numpy()
    if ts.size and not np.all(np.diff(ts) >= 0):
        viol += 1
print("Monotonic violations (groups):", viol)

# Approx Hz from ns timestamps
def est_hz_ns(ts_ns: pd.Series):
    arr = ts_ns.to_numpy()
    if arr.size < 3: return np.nan
    dt = np.diff(arr) / 1e9
    dt = dt[(dt > 0) & np.isfinite(dt)]
    return float(np.median(1.0 / dt)) if dt.size else np.nan

hz = capture24_df.groupby(["subject_id","session_id"])["timestamp_ns"].apply(est_hz_ns)
print(f"Median Hz: {np.nanmedian(hz.values):.2f} (target={SCHEMA['rate_hz']})")

# Required-not-null coverage
req = SCHEMA["expectations"]["required_not_null"]
pct = capture24_df[req].notnull().all(axis=1).mean() * 100
print(f"Rows meeting required-not-null: {pct:.2f}%")

# Mapping coverage
cov = (capture24_df["global_activity_id"] != UNKNOWN_ID).mean() * 100
print(f"Global mapping coverage: {cov:.1f}% (unknown={UNKNOWN_ID})")

print("\nTop-15 canonical labels:")
print(capture24_df["global_activity_label"].value_counts().head(15))


Subjects: 1 | Sessions: 1
Monotonic violations (groups): 0
Median Hz: 50.00 (target=50)
Rows meeting required-not-null: 100.00%
Global mapping coverage: 74.5% (unknown=9000)

Top-15 canonical labels:
global_activity_label
posture_stationary       2521361
other                    1275121
adl_household_general    1204418
adl_personal_care           9101
Name: count, dtype: Int64


### Step 5. Save outputs

In [ ]:
# =========================
# STEP 5 — Save outputs
# =========================
CLEANED.mkdir(parents=True, exist_ok=True)
out_path = CLEANED / "capture24.parquet"

capture24_df.to_parquet(out_path, index=False)
print("Wrote:", out_path)
